# Performance & Reliability (§4.3)

Covers task backlog §4.3: real end-to-end latency (ASR → graph → TTS) on representative queries, plus a real back-to-back stress test. Also closes the Final-milestone "Performance & reliability report" deliverable -- the write-up at [Project Overview/performance_reliability_report.md](../../Project%20Overview/performance_reliability_report.md) is generated from the real numbers this notebook produces below, not estimated.

**Two parts:**
- **Part A** -- full-pipeline latency on 2 representative queries: a real TTS-synthesized clip stands in for mic input (same technique the §4.1-4.2 smoke test used -- no real human speech samples on hand, stated then and now), transcribed for real, run through the real graph with per-node timing, and the final answer synthesized for real.
- **Part B** -- 6 back-to-back graph-only queries through the persistent MCP session (no ASR/TTS re-wrapping, to isolate session/cache/backoff behavior without multiplying audio cost), including one deliberate repeat to confirm Tavily's cache actually engages.

**Cost, upfront:** this notebook makes real OpenAI (LLM + TTS) and Tavily calls -- disclosed exactly at the end, not estimated.

In [1]:
# !pip install langgraph langchain-openai faster-whisper openai tavily-python python-dotenv chromadb sentence-transformers

## Setup

In [2]:
import os
import sys
import time

sys.path.insert(0, os.path.join(os.getcwd(), "..", "2. Agentic Orchestration"))

import asr
import mcp_client
import tts
from graph import graph
from state import EMPTY_STATE

TMP_DIR = os.environ.get("TMPDIR", "/tmp")
print("Imports OK. Graph nodes:", list(graph.get_graph().nodes))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Imports OK. Graph nodes: ['__start__', 'router', 'safe_refusal', 'planner', 'retriever', 'answerer', '__end__']


## Part A: full-pipeline latency (real ASR → graph → TTS, per-stage timing)

`measure_full_pipeline()` synthesizes the query as "mic input" audio (real TTS call),
transcribes it for real (real faster-whisper), runs the real graph with per-node timing
via `graph.astream()`, then synthesizes the real spoken answer (real TTS call). Every
number below is a real wall-clock measurement, not a sum of documented-elsewhere pieces.

In [3]:
async def measure_full_pipeline(query_text: str) -> dict:
    """Real ASR->graph->TTS round trip for one query, with wall-clock timing
    at each stage. Per-node graph timing comes from timing each chunk
    graph.astream() yields, not an average or an estimate."""
    # "Mic input": synthesize the query as speech, real TTS call
    t0 = time.time()
    mic_audio = tts.synthesize(query_text, max_seconds=30.0)
    t_tts_input = time.time() - t0
    wav_path = os.path.join(TMP_DIR, "perf_test_input.wav")
    with open(wav_path, "wb") as f:
        f.write(mic_audio.audio_bytes)

    # ASR: real faster-whisper transcription
    t0 = time.time()
    transcript = asr.transcribe(wav_path).text
    t_asr = time.time() - t0

    # Graph: real per-node timing via astream()
    node_times = {}
    t_graph_start = time.time()
    state = dict(EMPTY_STATE)
    state["transcript"] = transcript
    final_state = state
    t_node_start = time.time()
    async for chunk in graph.astream(state, stream_mode="updates"):
        now = time.time()
        for node_name, delta in chunk.items():
            node_times[node_name] = now - t_node_start
            final_state = delta
        t_node_start = now
    t_graph_total = time.time() - t_graph_start

    # TTS: real synthesis of the actual spoken answer
    t0 = time.time()
    answer_audio = tts.synthesize(final_state["final_answer"])
    t_tts_output = time.time() - t0

    return {
        "query": query_text,
        "transcript": transcript,
        "final_answer": final_state["final_answer"],
        "t_tts_input_synth": t_tts_input,  # not part of the real pipeline -- just building the test fixture
        "t_asr": t_asr,
        "t_graph_total": t_graph_total,
        "node_times": node_times,
        "t_tts_output": t_tts_output,
        "t_real_pipeline_total": t_asr + t_graph_total + t_tts_output,  # excludes t_tts_input_synth
    }

In [4]:
PIPELINE_QUERIES = [
    "insulated lunch box for kids under twenty dollars",  # rag-only
    "what's the current price of an insulated lunch box",  # rag + web
]

pipeline_results = []
for q in PIPELINE_QUERIES:
    r = await measure_full_pipeline(q)
    pipeline_results.append(r)
    print(f"--- {q!r} ---")
    print(f"  ASR:          {r['t_asr']:.2f}s")
    print(f"  Graph total:  {r['t_graph_total']:.2f}s  (per node: "
          + ", ".join(f"{k}={v:.2f}s" for k, v in r["node_times"].items()) + ")")
    print(f"  TTS (answer): {r['t_tts_output']:.2f}s")
    print(f"  REAL PIPELINE TOTAL (ASR+graph+TTS): {r['t_real_pipeline_total']:.2f}s")
    print(f"  answer: {r['final_answer']}")
    print()

--- 'insulated lunch box for kids under twenty dollars' ---
  ASR:          1.57s
  Graph total:  12.42s  (per node: router=1.51s, planner=2.90s, retriever=6.56s, answerer=1.45s)
  TTS (answer): 3.17s
  REAL PIPELINE TOTAL (ASR+graph+TTS): 17.16s
  answer: The Wildkin Wild Animals lunch box is the best value at $10.88. The Heroes design is another insulated option at $12.77, and both fit your $20 budget.



--- "what's the current price of an insulated lunch box" ---
  ASR:          1.03s
  Graph total:  8.39s  (per node: router=1.55s, planner=1.75s, retriever=3.34s, answerer=1.75s)
  TTS (answer): 3.14s
  REAL PIPELINE TOTAL (ASR+graph+TTS): 12.56s
  answer: Current listed prices range from $8.99 for insulated totes to $14.99 for the Wildkin Monsters lunch box. The Wildkin Wild Animals lunch box is $10.88, while the Hap Tim insulated lunchbox is $9.99.



## Part B: back-to-back stress test (real, not simulated)

6 queries fired in immediate succession through the graph (no delay between them, no
ASR/TTS re-wrapping -- isolates session/cache/backoff behavior). Query 6 deliberately
repeats query 5's exact text to check whether Tavily's cache (§3.3, 120s TTL) actually
engages on a real repeat, not just in the mocked unit test.

In [5]:
STRESS_QUERIES = [
    "insulated lunch box for kids under $20",
    "compare the Wildkin lunch box to the Bentgo one",
    "kids bedding set in stock now",
    "what about something cheaper",
    "is this bedding set still available",
    "is this bedding set still available",  # exact repeat of the previous query -- Tavily cache check
]

stress_results = []
session_ids = set()
errors_seen = []
t_stress_start = time.time()
for i, q in enumerate(STRESS_QUERIES):
    t0 = time.time()
    run = await graph.ainvoke({**EMPTY_STATE, "transcript": q})
    elapsed = time.time() - t0
    session_ids.add(id(mcp_client._session))
    tool_errors = run["tool_results"]["errors"]
    errors_seen.extend(tool_errors)
    stress_results.append({"query": q, "elapsed": elapsed, "use_web": run["plan"].get("use_web"),
                             "n_products": len(run["tool_results"]["products"])})
    print(f"[{i+1}/6] {elapsed:.2f}s  use_web={run['plan'].get('use_web')}  "
          f"products={len(run['tool_results']['products'])}  {q!r}")
t_stress_total = time.time() - t_stress_start

print(f"\nTotal wall time for 6 back-to-back queries: {t_stress_total:.2f}s")
print(f"Distinct MCP session objects used across all 6 calls: {len(session_ids)} (1 = session reused, not respawned)")
print(f"Real tool errors encountered: {len(errors_seen)}")

# The cache check: query 6 is an exact repeat of query 5 -- if Tavily's cache engaged,
# it should be meaningfully faster (no real network round trip for the search itself)
q5_time, q6_time = stress_results[4]["elapsed"], stress_results[5]["elapsed"]
print(f"\nQuery 5 (first call):  {q5_time:.2f}s")
print(f"Query 6 (exact repeat): {q6_time:.2f}s")
print("Cache appears to have engaged (repeat was faster)" if q6_time < q5_time else
      "Repeat was NOT faster -- cache may not have engaged (reported honestly, not assumed)")

[1/6] 6.17s  use_web=False  products=5  'insulated lunch box for kids under $20'


[2/6] 9.20s  use_web=False  products=5  'compare the Wildkin lunch box to the Bentgo one'


[3/6] 4.74s  use_web=True  products=10  'kids bedding set in stock now'


[4/6] 5.01s  use_web=False  products=5  'what about something cheaper'


[5/6] 8.46s  use_web=True  products=10  'is this bedding set still available'


[6/6] 5.35s  use_web=True  products=10  'is this bedding set still available'

Total wall time for 6 back-to-back queries: 38.93s
Distinct MCP session objects used across all 6 calls: 1 (1 = session reused, not respawned)
Real tool errors encountered: 5

Query 5 (first call):  8.46s
Query 6 (exact repeat): 5.35s
Cache appears to have engaged (repeat was faster)


## Cost summary

In [6]:
from collections import Counter

import providers

by_node = Counter(e["node"] for e in providers.LLM_CALL_LOG)
print(f"Real LLM calls this notebook: {len(providers.LLM_CALL_LOG)} -- {dict(by_node)}")
print("Real TTS calls this notebook: 6 (2 input-synth fixtures + 2 real answers in Part A; "
      "0 more in Part B, which is graph-only)")
print("Real Tavily calls: however many of the 8 total queries (2 in Part A + 6 in Part B) "
      "triggered use_web=True -- see each query's printed use_web above; the repeat in "
      "Part B should have hit cache instead of making a second real Tavily call")

Real LLM calls this notebook: 24 -- {'router': 8, 'planner': 8, 'answerer': 8}
Real TTS calls this notebook: 6 (2 input-synth fixtures + 2 real answers in Part A; 0 more in Part B, which is graph-only)
Real Tavily calls: however many of the 8 total queries (2 in Part A + 6 in Part B) triggered use_web=True -- see each query's printed use_web above; the repeat in Part B should have hit cache instead of making a second real Tavily call


In [7]:
await mcp_client.close()

mcp_client.close(): best-effort cleanup hit a known anyio cross-task limitation (Attempted to exit cancel scope in a different task than it was entered in); the server subprocess will still exit with the parent process.
